# NEWSLETTER ANALYSIS

### Import Libraries

In [1]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

# Display settings for better output viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## WEEKLY TABLE

### Load Reference Dates

In [2]:
# Step 2: Load the analysis reference date file
# This file contains the date ranges we need to analyze

ref_date_path = '../analysis_ref_date.csv'

# Read the file and extract Week Start Date and Week End Date
week_start_date = None
week_end_date = None

with open(ref_date_path, 'r') as f:
    lines = f.readlines()

for line in lines:
    # Look for lines containing "Week Start Date"
    if 'Week Start Date' in line and week_start_date is None:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            week_start_date = parts[1].strip()
    
    # Look for lines containing "Week End Date"
    if 'Week End Date' in line and week_end_date is None:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            week_end_date = parts[1].strip()
    
    # Break once we have both dates
    if week_start_date and week_end_date:
        break

# Display the extracted dates
print("Weekly Reference Dates Loaded:")
print(f"Week Start Date: {week_start_date}")
print(f"Week End Date: {week_end_date}")

# Create the "This Week" date range string (format: MM/DD/YYYY - MM/DD/YYYY)
this_week_range = f"{week_start_date} - {week_end_date}"
print(f"\nThis Week Range: {this_week_range}")

# Calculate "Previous Week" range
# Parse the dates to calculate previous week
this_week_start = pd.to_datetime(week_start_date, format='%m/%d/%Y')
this_week_end = pd.to_datetime(week_end_date, format='%m/%d/%Y')

# Previous week is 7 days before this week
prev_week_start = this_week_start - pd.Timedelta(days=7)
prev_week_end = this_week_end - pd.Timedelta(days=7)

# Format back to MM/DD/YYYY - MM/DD/YYYY
prev_week_range = f"{prev_week_start.strftime('%m/%d/%Y')} - {prev_week_end.strftime('%m/%d/%Y')}"
print(f"Previous Week Range: {prev_week_range}")

# Calculate 6-week average date range
# 6 weeks = from 5 weeks before this week to end of this week
# Note: Week runs Sunday to Saturday
six_weeks_start = this_week_start - pd.Timedelta(weeks=5)
six_weeks_end = this_week_end

six_week_range = f"{six_weeks_start.strftime('%m/%d/%Y')} - {six_weeks_end.strftime('%m/%d/%Y')}"
print(f"6-Week Average Range: {six_week_range}")

Weekly Reference Dates Loaded:
Week Start Date: 11/30/2025
Week End Date: 12/06/2025

This Week Range: 11/30/2025 - 12/06/2025
Previous Week Range: 11/23/2025 - 11/29/2025
6-Week Average Range: 10/26/2025 - 12/06/2025


### Define Helper Functions

In [3]:
# Step 3: Define helper functions for data extraction and calculation

def get_value_from_csv(csv_path, date_range, column_name):
    """
    Extract a value from a CSV file based on date range and column name
    
    Parameters:
    - csv_path: Path to the CSV file
    - date_range: Date range string to match (e.g., "11/30/2025 - 12/06/2025")
    - column_name: Name of the column to extract value from
    
    Returns:
    - The value if found, otherwise returns blank (empty string)
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_path)
        
        # Check if required columns exist
        if 'Date Range' not in df.columns or column_name not in df.columns:
            return ''
        
        # Filter by date range
        row = df[df['Date Range'] == date_range]
        
        # If row exists, get the value
        if not row.empty:
            value = row.iloc[0][column_name]
            # Check if value is NaN or blank
            if pd.isna(value) or value == '':
                return ''
            return value
        else:
            # Date range not found
            return ''
    
    except Exception as e:
        print(f"Error reading {csv_path}: {str(e)}")
        return ''


def get_value_from_csv_by_dates(csv_path, start_date, end_date, column_name):
    """
    Extract a value from a CSV file that uses separate 'Start Date' and 'End Date' columns
    
    This is specifically for files like 06_blended_weekly.csv that don't have a 'Date Range' column
    
    Parameters:
    - csv_path: Path to the CSV file
    - start_date: Start date string to match (e.g., "11/30/2025")
    - end_date: End date string to match (e.g., "12/06/2025")
    - column_name: Name of the column to extract value from
    
    Returns:
    - The value if found, otherwise returns blank
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_path)
        
        # Check if required columns exist
        if 'Start Date' not in df.columns or 'End Date' not in df.columns or column_name not in df.columns:
            return ''
        
        # Convert input dates from MM/DD/YYYY to datetime
        start_dt = pd.to_datetime(start_date, format='%m/%d/%Y')
        end_dt = pd.to_datetime(end_date, format='%m/%d/%Y')
        
        # Try multiple date formats for the CSV columns
        # The file might have YYYY-MM-DD or MM/DD/YYYY format
        date_formats_to_try = ['%Y-%m-%d', '%m/%d/%Y', '%Y/%m/%d']
        
        df_start_parsed = None
        df_end_parsed = None
        
        for date_format in date_formats_to_try:
            try:
                df_start_parsed = pd.to_datetime(df['Start Date'], format=date_format, errors='coerce')
                df_end_parsed = pd.to_datetime(df['End Date'], format=date_format, errors='coerce')
                
                # Check if parsing was successful (no NaT values)
                if not df_start_parsed.isna().all() and not df_end_parsed.isna().all():
                    break
            except:
                continue
        
        # If we couldn't parse the dates, return blank
        if df_start_parsed is None or df_end_parsed is None:
            return ''
        
        # Create temporary columns for comparison
        df['Start Date Parsed'] = df_start_parsed
        df['End Date Parsed'] = df_end_parsed
        
        # Filter by matching start and end dates
        row = df[(df['Start Date Parsed'] == start_dt) & (df['End Date Parsed'] == end_dt)]
        
        # If row exists, get the value
        if not row.empty:
            value = row.iloc[0][column_name]
            # Check if value is NaN or blank
            if pd.isna(value) or value == '':
                return ''
            
            # If it's a numeric value, add % sign for percentage columns
            if column_name in ['Open Rate', 'Click-Through Rate', 'Verified Click-Through Rate', 'Unsubscribe Rate']:
                if isinstance(value, (int, float)):
                    return f"{value}%"
            
            return value
        else:
            # Date range not found
            return ''
    
    except Exception as e:
        print(f"Error reading {csv_path}: {str(e)}")
        return ''


def get_value_from_excel(excel_path, sheet_name, date_range, column_name):
    """
    Extract a value from an Excel file based on date range and column name
    
    IMPORTANT: For beehiiv_unsubscribes.xlsx, the data is stored with Date Range in rows,
    not columns. This function handles that structure.
    
    Parameters:
    - excel_path: Path to the Excel file
    - sheet_name: Name of the sheet to read
    - date_range: Date range string to match (4-digit year format)
    - column_name: Name of the column to extract value from
    
    Returns:
    - The value if found, otherwise returns blank
    """
    try:
        # Read the Excel file
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        
        # Check if 'Date Range' exists as a column (row-based structure)
        if 'Date Range' in df.columns:
            # This is the beehiiv_unsubscribes.xlsx structure
            # Date ranges are in rows, metrics are in columns
            
            # Try exact match first
            row = df[df['Date Range'] == date_range]
            
            # If not found, try variations (with/without leading zeros, 2-digit vs 4-digit year)
            if row.empty:
                # Try converting date_range format
                # From "11/30/2025 - 12/06/2025" to "11/30/2025 - 12/6/2025" (no leading zero on day)
                # Or vice versa
                date_range_variations = [date_range]
                
                # Add variation without leading zeros on days
                parts = date_range.split(' - ')
                if len(parts) == 2:
                    start_parts = parts[0].split('/')
                    end_parts = parts[1].split('/')
                    # Remove leading zeros from days
                    start_no_zero = f"{start_parts[0]}/{int(start_parts[1])}/{start_parts[2]}"
                    end_no_zero = f"{end_parts[0]}/{int(end_parts[1])}/{end_parts[2]}"
                    date_range_variations.append(f"{start_no_zero} - {end_no_zero}")
                
                # Try each variation
                for variation in date_range_variations:
                    row = df[df['Date Range'] == variation]
                    if not row.empty:
                        break
            
            if row.empty:
                return ''
            
            # Check if column exists
            if column_name not in df.columns:
                return ''
            
            # Get the value
            value = row.iloc[0][column_name]
            
            if pd.isna(value) or value == '':
                return ''
            
            return value
        else:
            # This shouldn't happen for our current use case, but keeping for compatibility
            return ''
    
    except Exception as e:
        print(f"Error reading {excel_path}: {str(e)}")
        return ''


def calculate_wow_change(current_value, previous_value):
    """
    Calculate Week-over-Week percentage change
    
    Formula: ((Current - Previous) / Previous) * 100
    
    Parameters:
    - current_value: Value for this week
    - previous_value: Value for previous week
    
    Returns:
    - Formatted string with + or - sign and % symbol
    - Returns blank if calculation cannot be performed
    """
    try:
        # Check if either value is blank/empty
        if current_value == '' or previous_value == '' or current_value is None or previous_value is None:
            return ''
        
        # Convert to float (handle percentage strings like "3.58%")
        if isinstance(current_value, str):
            current_value = current_value.replace('%', '').strip()
            if current_value == '':
                return ''
            current_value = float(current_value)
        
        if isinstance(previous_value, str):
            previous_value = previous_value.replace('%', '').strip()
            if previous_value == '':
                return ''
            previous_value = float(previous_value)
        
        # Avoid division by zero
        if previous_value == 0:
            return ''
        
        # Calculate percentage change
        wow_change = ((current_value - previous_value) / previous_value) * 100
        
        # Format with + or - sign
        if wow_change >= 0:
            return f"+{wow_change:.2f}%"
        else:
            return f"{wow_change:.2f}%"
    
    except Exception as e:
        print(f"Error calculating WoW change: {str(e)}")
        return ''


def calculate_division_percentage(numerator, denominator):
    """
    Safely divide two values and return as percentage
    
    Formula: (numerator / denominator) * 100
    
    Parameters:
    - numerator: Top value
    - denominator: Bottom value
    
    Returns:
    - Result as percentage string with % symbol
    - Returns blank if calculation cannot be performed
    """
    try:
        # Check if either value is blank
        if numerator == '' or denominator == '' or numerator is None or denominator is None:
            return ''
        
        # Convert to float
        if isinstance(numerator, str):
            numerator = float(numerator.replace('%', '').replace(',', '').strip())
        if isinstance(denominator, str):
            denominator = float(denominator.replace('%', '').replace(',', '').strip())
        
        # Avoid division by zero
        if denominator == 0:
            return ''
        
        # Calculate percentage
        result = (numerator / denominator) * 100
        
        return f"{result:.2f}%"
    
    except Exception as e:
        print(f"Error in division: {str(e)}")
        return ''


def convert_decimal_to_percentage(value):
    """
    Convert decimal value to percentage string
    
    Example: 0.0192 becomes "1.92%"
    
    Parameters:
    - value: Decimal value
    
    Returns:
    - Formatted percentage string
    """
    try:
        if value == '' or value is None or pd.isna(value):
            return ''
        
        # Convert to float if string
        if isinstance(value, str):
            value = value.replace('%', '').strip()
            if value == '':
                return ''
            value = float(value)
        
        # Convert decimal to percentage
        percentage = value * 100
        
        return f"{percentage:.2f}%"
    
    except Exception as e:
        print(f"Error converting to percentage: {str(e)}")
        return ''


def calculate_6week_average(csv_path, column_name, end_date, use_start_end_dates=False):
    """
    Calculate 6-week average for a metric
    
    Note: Sums or averages values from the 6 weeks ending on end_date,
    then divides by 6 for the average
    
    Parameters:
    - csv_path: Path to the CSV file
    - column_name: Column to calculate average for
    - end_date: End date of the 6-week period (datetime object)
    - use_start_end_dates: If True, uses 'Start Date' and 'End Date' columns instead of 'Date Range'
    
    Returns:
    - Average value or blank if cannot calculate
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_path)
        
        if column_name not in df.columns:
            return ''
        
        # Parse dates based on file structure
        if use_start_end_dates:
            # For files with 'Start Date' and 'End Date' columns
            if 'End Date' not in df.columns:
                return ''
            
            # Try multiple date formats
            date_formats_to_try = ['%Y-%m-%d', '%m/%d/%Y', '%Y/%m/%d']
            
            df_end_parsed = None
            for date_format in date_formats_to_try:
                try:
                    df_end_parsed = pd.to_datetime(df['End Date'], format=date_format, errors='coerce')
                    if not df_end_parsed.isna().all():
                        break
                except:
                    continue
            
            if df_end_parsed is None:
                return ''
            
            df['End Date Parsed'] = df_end_parsed
        else:
            # For files with 'Date Range' column
            if 'Date Range' not in df.columns:
                return ''
            # Parse Date Range to get end dates
            # Assuming Date Range format is "MM/DD/YYYY - MM/DD/YYYY"
            df['End Date Parsed'] = df['Date Range'].apply(
                lambda x: pd.to_datetime(x.split(' - ')[1], format='%m/%d/%Y', errors='coerce') if ' - ' in str(x) else None
            )
        
        # Calculate start date for 6-week period
        start_date = end_date - pd.Timedelta(weeks=5)
        
        # Filter for weeks within the 6-week period
        mask = (df['End Date Parsed'] >= start_date) & (df['End Date Parsed'] <= end_date)
        weeks_data = df[mask]
        
        if weeks_data.empty:
            return ''
        
        # Get values for the column
        values = []
        for val in weeks_data[column_name]:
            if val != '' and not pd.isna(val):
                # Handle percentage strings
                if isinstance(val, str):
                    val = val.replace('%', '').replace(',', '').strip()
                    if val != '':
                        values.append(float(val))
                else:
                    values.append(float(val))
        
        if len(values) == 0:
            return ''
        
        # Calculate average
        average = sum(values) / len(values)
        
        return round(average, 2)
    
    except Exception as e:
        print(f"Error calculating 6-week average: {str(e)}")
        return ''


def calculate_6week_average_from_excel(excel_path, sheet_name, column_name, end_date):
    """
    Calculate 6-week average for a metric from Excel file with Date Range column
    
    Parameters:
    - excel_path: Path to the Excel file
    - sheet_name: Name of the sheet
    - column_name: Column to calculate average for
    - end_date: End date of the 6-week period (datetime object)
    
    Returns:
    - Average value or blank if cannot calculate
    """
    try:
        # Read the Excel file
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        
        if 'Date Range' not in df.columns or column_name not in df.columns:
            return ''
        
        # Parse Date Range to get end dates
        # Format: "MM/DD/YYYY - MM/DD/YYYY"
        def parse_end_date(date_range_str):
            try:
                if ' - ' in str(date_range_str):
                    end_part = date_range_str.split(' - ')[1]
                    return pd.to_datetime(end_part, format='%m/%d/%Y', errors='coerce')
                return None
            except:
                return None
        
        df['End Date Parsed'] = df['Date Range'].apply(parse_end_date)
        
        # Calculate start date for 6-week period (5 weeks back + current week = 6 weeks)
        start_date = end_date - pd.Timedelta(weeks=5)
        
        # Filter for weeks within the 6-week period
        mask = (df['End Date Parsed'] >= start_date) & (df['End Date Parsed'] <= end_date)
        weeks_data = df[mask]
        
        if weeks_data.empty:
            return ''
        
        # Get values for the column
        values = []
        for val in weeks_data[column_name]:
            if val != '' and not pd.isna(val):
                # Handle percentage strings
                if isinstance(val, str):
                    val = val.replace('%', '').replace(',', '').strip()
                    if val != '':
                        values.append(float(val))
                else:
                    values.append(float(val))
        
        if len(values) == 0:
            return ''
        
        # Calculate average
        average = sum(values) / len(values)
        
        return round(average, 2)
    
    except Exception as e:
        print(f"Error calculating 6-week average from Excel: {str(e)}")
        return ''


print("✓ Helper functions defined successfully!")

✓ Helper functions defined successfully!


### Extract Weekly Data

In [4]:
# Step 4: Extract data for This Week and Previous Week

print("=" * 70)
print("EXTRACTING WEEKLY NEWSLETTER DATA")
print("=" * 70)

# Define all data source paths
csv_sources = {
    'newsletter': '../outputs/newsletter/weekly_newsletter.csv',
    'general_newsletter': '../outputs/general_newsletter/weekly_table.csv',
    'newsletter_series_blended': '../outputs/newsletter_series/06_blended_weekly.csv',
}

excel_sources = {
    'beehiiv_unsubscribes': '../data/beehiiv_unsubscribes.xlsx'
}

# Initialize dictionary to store weekly data
weekly_data = {}

print(f"\nThis Week: {this_week_range}")
print(f"Previous Week: {prev_week_range}")
print(f"6-Week Range: {six_week_range}")
print("\n" + "-" * 70)

# 1. Visits (from newsletter/weekly_newsletter.csv)
print("\n1. Extracting Visits...")
this_visits = get_value_from_csv(csv_sources['newsletter'], this_week_range, 'Visits')
prev_visits = get_value_from_csv(csv_sources['newsletter'], prev_week_range, 'Visits')
avg_6wk_visits = calculate_6week_average(csv_sources['newsletter'], 'Visits', this_week_end)
weekly_data['Visits (Newsletter Landing Page Visits)'] = {
    'this_week': this_visits,
    'prev_week': prev_visits,
    'six_week_avg': avg_6wk_visits
}
print(f"   This Week: {this_visits}")
print(f"   Previous Week: {prev_visits}")
print(f"   6-Week Average: {avg_6wk_visits}")

# 2. New Subscribers (from general_newsletter/weekly_table.csv)
print("\n2. Extracting New Subscribers...")
this_new_subs = get_value_from_csv(csv_sources['general_newsletter'], this_week_range, 'Subscribed')
prev_new_subs = get_value_from_csv(csv_sources['general_newsletter'], prev_week_range, 'Subscribed')
avg_6wk_new_subs = calculate_6week_average(csv_sources['general_newsletter'], 'Subscribed', this_week_end)
weekly_data['New Subscribers'] = {
    'this_week': this_new_subs,
    'prev_week': prev_new_subs,
    'six_week_avg': avg_6wk_new_subs
}
print(f"   This Week: {this_new_subs}")
print(f"   Previous Week: {prev_new_subs}")
print(f"   6-Week Average: {avg_6wk_new_subs}")

# 3. CVR % (Calculated: New Subscribers / Visits)
print("\n3. Calculating Join LP CVR %...")
this_cvr = calculate_division_percentage(this_new_subs, this_visits)
prev_cvr = calculate_division_percentage(prev_new_subs, prev_visits)
# For 6-week average CVR, divide average new subs by average visits
avg_6wk_cvr = calculate_division_percentage(avg_6wk_new_subs, avg_6wk_visits) if avg_6wk_new_subs != '' and avg_6wk_visits != '' else ''
weekly_data['Join LP CVR %'] = {
    'this_week': this_cvr,
    'prev_week': prev_cvr,
    'six_week_avg': avg_6wk_cvr
}
print(f"   This Week: {this_cvr}")
print(f"   Previous Week: {prev_cvr}")
print(f"   6-Week Average: {avg_6wk_cvr}")

# 4. Active Unsubs (from data/beehiiv_unsubscribes.xlsx)
print("\n4. Extracting Active Unsubs...")
this_active_unsubs = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', this_week_range, 'Active Unsubs')
prev_active_unsubs = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', prev_week_range, 'Active Unsubs')
# For 6-week average, get the total from the 6-week range row and divide by 6
six_week_total_active = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', six_week_range, 'Active Unsubs')
if six_week_total_active != '' and six_week_total_active is not None:
    avg_6wk_active_unsubs = round(float(six_week_total_active) / 6, 2)
else:
    avg_6wk_active_unsubs = ''
weekly_data['Active Unsubs'] = {
    'this_week': this_active_unsubs,
    'prev_week': prev_active_unsubs,
    'six_week_avg': avg_6wk_active_unsubs
}
print(f"   This Week: {this_active_unsubs}")
print(f"   Previous Week: {prev_active_unsubs}")
print(f"   6-Week Average: {avg_6wk_active_unsubs}")

# 5. Auto Unsubs (from data/beehiiv_unsubscribes.xlsx)
print("\n5. Extracting Auto Unsubs...")
this_auto_unsubs = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', this_week_range, 'Auto Unsubs')
prev_auto_unsubs = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', prev_week_range, 'Auto Unsubs')
# For 6-week average, get the total from the 6-week range row and divide by 6
six_week_total_auto = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', six_week_range, 'Auto Unsubs')
if six_week_total_auto != '' and six_week_total_auto is not None:
    avg_6wk_auto_unsubs = round(float(six_week_total_auto) / 6, 2)
else:
    avg_6wk_auto_unsubs = ''
weekly_data['Auto Unsubs'] = {
    'this_week': this_auto_unsubs,
    'prev_week': prev_auto_unsubs,
    'six_week_avg': avg_6wk_auto_unsubs
}
print(f"   This Week: {this_auto_unsubs}")
print(f"   Previous Week: {prev_auto_unsubs}")
print(f"   6-Week Average: {avg_6wk_auto_unsubs}")

# 6. Unsubscribes (from data/beehiiv_unsubscribes.xlsx)
print("\n6. Extracting Unsubscribes...")
this_unsubs = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', this_week_range, 'Total Unsubs')
prev_unsubs = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', prev_week_range, 'Total Unsubs')
# For 6-week average, get the total from the 6-week range row and divide by 6
six_week_total_unsubs = get_value_from_excel(excel_sources['beehiiv_unsubscribes'], 'Sheet1', six_week_range, 'Total Unsubs')
if six_week_total_unsubs != '' and six_week_total_unsubs is not None:
    avg_6wk_unsubs = round(float(six_week_total_unsubs) / 6, 2)
else:
    avg_6wk_unsubs = ''
weekly_data['Unsubscribes'] = {
    'this_week': this_unsubs,
    'prev_week': prev_unsubs,
    'six_week_avg': avg_6wk_unsubs
}
print(f"   This Week: {this_unsubs}")
print(f"   Previous Week: {prev_unsubs}")
print(f"   6-Week Average: {avg_6wk_unsubs}")

# 7. Leak Rate (Calculated: Unsubscribes / New Subscribers)
print("\n7. Calculating Leak Rate...")
this_leak_rate = calculate_division_percentage(this_unsubs, this_new_subs)
prev_leak_rate = calculate_division_percentage(prev_unsubs, prev_new_subs)
avg_6wk_leak_rate = calculate_division_percentage(avg_6wk_unsubs, avg_6wk_new_subs)
weekly_data['Leak Rate'] = {
    'this_week': this_leak_rate,
    'prev_week': prev_leak_rate,
    'six_week_avg': avg_6wk_leak_rate
}
print(f"   This Week: {this_leak_rate}")
print(f"   Previous Week: {prev_leak_rate}")
print(f"   6-Week Average: {avg_6wk_leak_rate}")

# 8. Net New Subscribers (from general_newsletter/weekly_table.csv)
print("\n8. Extracting Net New Subscribers...")
this_net_new = get_value_from_csv(csv_sources['general_newsletter'], this_week_range, 'Net')
prev_net_new = get_value_from_csv(csv_sources['general_newsletter'], prev_week_range, 'Net')
avg_6wk_net_new = calculate_6week_average(csv_sources['general_newsletter'], 'Net', this_week_end)
weekly_data['Net New Subscribers'] = {
    'this_week': this_net_new,
    'prev_week': prev_net_new,
    'six_week_avg': avg_6wk_net_new
}
print(f"   This Week: {this_net_new}")
print(f"   Previous Week: {prev_net_new}")
print(f"   6-Week Average: {avg_6wk_net_new}")

# 9. Total Subscribers (from general_newsletter/weekly_table.csv)
print("\n9. Extracting Total Subscribers...")
this_total_subs = get_value_from_csv(csv_sources['general_newsletter'], this_week_range, 'Active Subscribers')
prev_total_subs = get_value_from_csv(csv_sources['general_newsletter'], prev_week_range, 'Active Subscribers')
# Calculate 6-week average internally (for use in List Unsub Rate calculation)
# But display it as blank in the table
avg_6wk_total_subs_internal = calculate_6week_average(
    csv_sources['general_newsletter'],
    'Active Subscribers',
    this_week_end,
    use_start_end_dates=False
)
weekly_data['Total Subscribers'] = {
    'this_week': this_total_subs,
    'prev_week': prev_total_subs,
    'six_week_avg': ''  # Display as blank in the table
}
print(f"   This Week: {this_total_subs}")
print(f"   Previous Week: {prev_total_subs}")
print(f"   6-Week Average: (calculated internally but not displayed)")

# 10. List Unsub Rate (Calculated: Unsubscribes / Total Subscribers)
print("\n10. Calculating List Unsub Rate...")
this_list_unsub_rate = calculate_division_percentage(this_unsubs, this_total_subs)
prev_list_unsub_rate = calculate_division_percentage(prev_unsubs, prev_total_subs)
# Calculate using the internal 6-week averages
avg_6wk_list_unsub_rate = calculate_division_percentage(avg_6wk_unsubs, avg_6wk_total_subs_internal)
weekly_data['List Unsub Rate'] = {
    'this_week': this_list_unsub_rate,
    'prev_week': prev_list_unsub_rate,
    'six_week_avg': avg_6wk_list_unsub_rate
}
print(f"   This Week: {this_list_unsub_rate}")
print(f"   Previous Week: {prev_list_unsub_rate}")
print(f"   6-Week Average: {avg_6wk_list_unsub_rate}")

# 11. Growth Rate (from general_newsletter/weekly_table.csv - convert decimal to percentage)
print("\n11. Extracting Growth Rate (%)...")
this_growth_raw = get_value_from_csv(csv_sources['general_newsletter'], this_week_range, 'Growth Rate')
prev_growth_raw = get_value_from_csv(csv_sources['general_newsletter'], prev_week_range, 'Growth Rate')
this_growth = convert_decimal_to_percentage(this_growth_raw)
prev_growth = convert_decimal_to_percentage(prev_growth_raw)
# For 6-week average, get average of the decimal values then convert
avg_6wk_growth_raw = calculate_6week_average(csv_sources['general_newsletter'], 'Growth Rate', this_week_end)
avg_6wk_growth = convert_decimal_to_percentage(avg_6wk_growth_raw) if avg_6wk_growth_raw != '' else ''
weekly_data['Growth Rate (%)'] = {
    'this_week': this_growth,
    'prev_week': prev_growth,
    'six_week_avg': avg_6wk_growth
}
print(f"   This Week: {this_growth}")
print(f"   Previous Week: {prev_growth}")
print(f"   6-Week Average: {avg_6wk_growth}")

# 12. Blended Open Rate (from newsletter_series/06_blended_weekly.csv)
# NOTE: This CSV uses 'Start Date' and 'End Date' columns instead of 'Date Range'
print("\n12. Extracting Blended Open Rate...")
this_open_rate = get_value_from_csv_by_dates(
    csv_sources['newsletter_series_blended'], 
    week_start_date, 
    week_end_date, 
    'Open Rate'
)
prev_open_rate = get_value_from_csv_by_dates(
    csv_sources['newsletter_series_blended'], 
    prev_week_start.strftime('%m/%d/%Y'), 
    prev_week_end.strftime('%m/%d/%Y'), 
    'Open Rate'
)
avg_6wk_open_rate_raw = calculate_6week_average(
    csv_sources['newsletter_series_blended'], 
    'Open Rate', 
    this_week_end,
    use_start_end_dates=True  # ← Make sure this is TRUE
)
# Convert numeric to percentage if needed
if avg_6wk_open_rate_raw != '' and isinstance(avg_6wk_open_rate_raw, (int, float)):
    avg_6wk_open_rate = f"{avg_6wk_open_rate_raw}%"
else:
    avg_6wk_open_rate = avg_6wk_open_rate_raw if avg_6wk_open_rate_raw != '' else ''

weekly_data['Blended Open Rate'] = {
    'this_week': this_open_rate,
    'prev_week': prev_open_rate,
    'six_week_avg': avg_6wk_open_rate
}
print(f"   This Week: {this_open_rate}")
print(f"   Previous Week: {prev_open_rate}")
print(f"   6-Week Average: {avg_6wk_open_rate}")

# 13. Blended Click Rate (from newsletter_series/06_blended_weekly.csv)
# NOTE: This CSV uses 'Start Date' and 'End Date' columns instead of 'Date Range'
print("\n13. Extracting Blended Click Rate...")
this_click_rate = get_value_from_csv_by_dates(
    csv_sources['newsletter_series_blended'], 
    week_start_date, 
    week_end_date, 
    'Verified Click-Through Rate'
)
prev_click_rate = get_value_from_csv_by_dates(
    csv_sources['newsletter_series_blended'], 
    prev_week_start.strftime('%m/%d/%Y'), 
    prev_week_end.strftime('%m/%d/%Y'), 
    'Verified Click-Through Rate'
)
avg_6wk_click_rate_raw = calculate_6week_average(
    csv_sources['newsletter_series_blended'], 
    'Verified Click-Through Rate', 
    this_week_end,
    use_start_end_dates=True  # ← Make sure this is TRUE
)
# Convert numeric to percentage if needed
if avg_6wk_click_rate_raw != '' and isinstance(avg_6wk_click_rate_raw, (int, float)):
    avg_6wk_click_rate = f"{avg_6wk_click_rate_raw}%"
else:
    avg_6wk_click_rate = avg_6wk_click_rate_raw if avg_6wk_click_rate_raw != '' else ''

weekly_data['Blended Click Rate'] = {
    'this_week': this_click_rate,
    'prev_week': prev_click_rate,
    'six_week_avg': avg_6wk_click_rate
}
print(f"   This Week: {this_click_rate}")
print(f"   Previous Week: {prev_click_rate}")
print(f"   6-Week Average: {avg_6wk_click_rate}")

# 14. Blended Unsub Rate (from newsletter_series/06_blended_weekly.csv)
# NOTE: This CSV uses 'Start Date' and 'End Date' columns instead of 'Date Range'
print("\n14. Extracting Blended Unsub Rate...")
this_blended_unsub = get_value_from_csv_by_dates(
    csv_sources['newsletter_series_blended'], 
    week_start_date, 
    week_end_date, 
    'Unsubscribe Rate'
)
prev_blended_unsub = get_value_from_csv_by_dates(
    csv_sources['newsletter_series_blended'], 
    prev_week_start.strftime('%m/%d/%Y'), 
    prev_week_end.strftime('%m/%d/%Y'), 
    'Unsubscribe Rate'
)
avg_6wk_blended_unsub_raw = calculate_6week_average(
    csv_sources['newsletter_series_blended'], 
    'Unsubscribe Rate', 
    this_week_end,
    use_start_end_dates=True  # ← Make sure this is TRUE
)
# Convert numeric to percentage if needed
if avg_6wk_blended_unsub_raw != '' and isinstance(avg_6wk_blended_unsub_raw, (int, float)):
    avg_6wk_blended_unsub = f"{avg_6wk_blended_unsub_raw}%"
else:
    avg_6wk_blended_unsub = avg_6wk_blended_unsub_raw if avg_6wk_blended_unsub_raw != '' else ''

weekly_data['Blended Unsub Rate'] = {
    'this_week': this_blended_unsub,
    'prev_week': prev_blended_unsub,
    'six_week_avg': avg_6wk_blended_unsub
}
print(f"   This Week: {this_blended_unsub}")
print(f"   Previous Week: {prev_blended_unsub}")
print(f"   6-Week Average: {avg_6wk_blended_unsub}")

EXTRACTING WEEKLY NEWSLETTER DATA

This Week: 11/30/2025 - 12/06/2025
Previous Week: 11/23/2025 - 11/29/2025
6-Week Range: 10/26/2025 - 12/06/2025

----------------------------------------------------------------------

1. Extracting Visits...
   This Week: 37997
   Previous Week: 33821
   6-Week Average: 29825.0

2. Extracting New Subscribers...
   This Week: 4750
   Previous Week: 4477
   6-Week Average: 4092.67

3. Calculating Join LP CVR %...
   This Week: 12.50%
   Previous Week: 13.24%
   6-Week Average: 13.72%

4. Extracting Active Unsubs...
   This Week: 2349
   Previous Week: 356
   6-Week Average: 303.83

5. Extracting Auto Unsubs...
   This Week: 1
   Previous Week: 3099
   6-Week Average: 29.5

6. Extracting Unsubscribes...
   This Week: 2350
   Previous Week: 3455
   6-Week Average: 333.33

7. Calculating Leak Rate...
   This Week: 49.47%
   Previous Week: 77.17%
   6-Week Average: 8.14%

8. Extracting Net New Subscribers...
   This Week: 2454
   Previous Week: 2540
   6-W

### Diagnose Blank Values

In [5]:
# Step 4.5: Diagnose why certain weekly values are blank

print(f"\n{'='*80}")
print("DIAGNOSING BLANK VALUES FOR WEEKLY DATA")
print(f"{'='*80}\n")

def diagnose_weekly_blank_csv(csv_path, date_range, column_name, metric_name):
    """
    Diagnose why a weekly CSV value is blank
    """
    diagnosis = {
        'metric': metric_name,
        'file': csv_path,
        'issue': None,
        'details': None
    }
    
    try:
        if not os.path.exists(csv_path):
            diagnosis['issue'] = 'FILE_NOT_FOUND'
            diagnosis['details'] = f"File does not exist: {csv_path}"
            return diagnosis
        
        df = pd.read_csv(csv_path)
        
        if 'Date Range' not in df.columns:
            diagnosis['issue'] = 'MISSING_DATE_RANGE_COLUMN'
            diagnosis['details'] = f"'Date Range' column not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        if column_name not in df.columns:
            diagnosis['issue'] = 'MISSING_DATA_COLUMN'
            diagnosis['details'] = f"Column '{column_name}' not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        row = df[df['Date Range'] == date_range]
        if row.empty:
            diagnosis['issue'] = 'DATE_RANGE_NOT_FOUND'
            diagnosis['details'] = f"Date range '{date_range}' not found. Available ranges: {df['Date Range'].tolist()}"
            return diagnosis
        
        value = row.iloc[0][column_name]
        if pd.isna(value) or value == '':
            diagnosis['issue'] = 'BLANK_CELL_VALUE'
            diagnosis['details'] = f"Cell exists but value is empty/blank for '{date_range}'"
            return diagnosis
        
        diagnosis['issue'] = 'NO_ISSUE'
        diagnosis['details'] = f"Value found: {value}"
        return diagnosis
        
    except Exception as e:
        diagnosis['issue'] = 'ERROR'
        diagnosis['details'] = f"Error: {str(e)}"
        return diagnosis


def diagnose_weekly_blank_csv_by_dates(csv_path, start_date, end_date, column_name, metric_name):
    """
    Diagnose why a weekly CSV value is blank for files using Start Date and End Date columns
    """
    diagnosis = {
        'metric': metric_name,
        'file': csv_path,
        'issue': None,
        'details': None
    }
    
    try:
        if not os.path.exists(csv_path):
            diagnosis['issue'] = 'FILE_NOT_FOUND'
            diagnosis['details'] = f"File does not exist: {csv_path}"
            return diagnosis
        
        df = pd.read_csv(csv_path)
        
        if 'Start Date' not in df.columns or 'End Date' not in df.columns:
            diagnosis['issue'] = 'MISSING_DATE_COLUMNS'
            diagnosis['details'] = f"'Start Date' or 'End Date' columns not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        if column_name not in df.columns:
            diagnosis['issue'] = 'MISSING_DATA_COLUMN'
            diagnosis['details'] = f"Column '{column_name}' not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        # Convert dates for comparison
        df['Start Date'] = pd.to_datetime(df['Start Date'], format='%m/%d/%Y', errors='coerce')
        df['End Date'] = pd.to_datetime(df['End Date'], format='%m/%d/%Y', errors='coerce')
        start_dt = pd.to_datetime(start_date, format='%m/%d/%Y')
        end_dt = pd.to_datetime(end_date, format='%m/%d/%Y')
        
        row = df[(df['Start Date'] == start_dt) & (df['End Date'] == end_dt)]
        if row.empty:
            diagnosis['issue'] = 'DATE_RANGE_NOT_FOUND'
            diagnosis['details'] = f"Date range '{start_date} - {end_date}' not found in file"
            return diagnosis
        
        value = row.iloc[0][column_name]
        if pd.isna(value) or value == '':
            diagnosis['issue'] = 'BLANK_CELL_VALUE'
            diagnosis['details'] = f"Cell exists but value is empty/blank"
            return diagnosis
        
        diagnosis['issue'] = 'NO_ISSUE'
        diagnosis['details'] = f"Value found: {value}"
        return diagnosis
        
    except Exception as e:
        diagnosis['issue'] = 'ERROR'
        diagnosis['details'] = f"Error: {str(e)}"
        return diagnosis


def diagnose_weekly_blank_excel(excel_path, sheet_name, date_range, column_name, metric_name):
    """
    Diagnose why a weekly Excel value is blank
    For beehiiv_unsubscribes.xlsx, date ranges are stored in ROWS (not columns)
    """
    diagnosis = {
        'metric': metric_name,
        'file': excel_path,
        'issue': None,
        'details': None
    }
    
    try:
        if not os.path.exists(excel_path):
            diagnosis['issue'] = 'FILE_NOT_FOUND'
            diagnosis['details'] = f"File does not exist: {excel_path}"
            return diagnosis
        
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        
        # Check if 'Date Range' exists as a column (row-based structure)
        if 'Date Range' not in df.columns:
            diagnosis['issue'] = 'MISSING_DATE_RANGE_COLUMN'
            diagnosis['details'] = f"'Date Range' column not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        if column_name not in df.columns:
            diagnosis['issue'] = 'MISSING_DATA_COLUMN'
            diagnosis['details'] = f"Column '{column_name}' not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        # Try to find the date range in rows
        row = df[df['Date Range'] == date_range]
        
        # If not found, try variations
        if row.empty:
            # Try without leading zeros
            parts = date_range.split(' - ')
            if len(parts) == 2:
                start_parts = parts[0].split('/')
                end_parts = parts[1].split('/')
                start_no_zero = f"{start_parts[0]}/{int(start_parts[1])}/{start_parts[2]}"
                end_no_zero = f"{end_parts[0]}/{int(end_parts[1])}/{end_parts[2]}"
                date_range_variation = f"{start_no_zero} - {end_no_zero}"
                row = df[df['Date Range'] == date_range_variation]
        
        if row.empty:
            diagnosis['issue'] = 'DATE_RANGE_NOT_FOUND'
            diagnosis['details'] = f"Date range '{date_range}' not found in rows. Available ranges: {df['Date Range'].tolist()}"
            return diagnosis
        
        value = row.iloc[0][column_name]
        if pd.isna(value) or value == '':
            diagnosis['issue'] = 'BLANK_CELL_VALUE'
            diagnosis['details'] = f"Cell exists but value is empty/blank"
            return diagnosis
        
        diagnosis['issue'] = 'NO_ISSUE'
        diagnosis['details'] = f"Value found: {value}"
        return diagnosis
        
    except Exception as e:
        diagnosis['issue'] = 'ERROR'
        diagnosis['details'] = f"Error: {str(e)}"
        return diagnosis


# Check metrics with blank values
blank_metrics = []
issue_summary = {}

print("Checking PREVIOUS WEEK data sources:")
print("-" * 80)

# Map metrics to their sources
metric_sources = {
    'Visits (Prev)': ('csv', csv_sources['newsletter'], 'Visits', 'date_range'),
    'New Subscribers (Prev)': ('csv', csv_sources['general_newsletter'], 'Subscribed', 'date_range'),
    'Active Unsubs (Prev)': ('excel', excel_sources['beehiiv_unsubscribes'], 'Active Unsubs', 'date_range'),
    'Auto Unsubs (Prev)': ('excel', excel_sources['beehiiv_unsubscribes'], 'Auto Unsubs', 'date_range'),
    'Unsubscribes (Prev)': ('excel', excel_sources['beehiiv_unsubscribes'], 'Total Unsubs', 'date_range'),
    'Net New Subscribers (Prev)': ('csv', csv_sources['general_newsletter'], 'Net', 'date_range'),
    'Total Subscribers (Prev)': ('csv', csv_sources['general_newsletter'], 'Active Subscribers', 'date_range'),
    'Growth Rate (Prev)': ('csv', csv_sources['general_newsletter'], 'Growth Rate', 'date_range'),
    'Blended Open Rate (Prev)': ('csv', csv_sources['newsletter_series_blended'], 'Open Rate', 'start_end_dates'),
    'Blended Click Rate (Prev)': ('csv', csv_sources['newsletter_series_blended'], 'Verified Click-Through Rate', 'start_end_dates'),
    'Blended Unsub Rate (Prev)': ('csv', csv_sources['newsletter_series_blended'], 'Unsubscribe Rate', 'start_end_dates'),
}

for metric_name, source_info in metric_sources.items():
    source_type = source_info[0]
    source_path = source_info[1]
    column_name = source_info[2]
    date_type = source_info[3]
    
    if source_type == 'csv' and date_type == 'date_range':
        diagnosis = diagnose_weekly_blank_csv(source_path, prev_week_range, column_name, metric_name)
    elif source_type == 'csv' and date_type == 'start_end_dates':
        diagnosis = diagnose_weekly_blank_csv_by_dates(
            source_path, 
            prev_week_start.strftime('%m/%d/%Y'), 
            prev_week_end.strftime('%m/%d/%Y'), 
            column_name, 
            metric_name
        )
    else:  # excel
        diagnosis = diagnose_weekly_blank_excel(source_path, 'Sheet1', prev_week_range, column_name, metric_name)
    
    if diagnosis['issue'] not in ['NO_ISSUE']:
        blank_metrics.append(diagnosis)
        issue_summary[diagnosis['issue']] = issue_summary.get(diagnosis['issue'], 0) + 1
        
        print(f"❌ {diagnosis['metric']}: {diagnosis['issue']}")
        print(f"   → {diagnosis['details']}\n")

print("\n" + "=" * 80)
print("Checking THIS WEEK data sources:")
print("-" * 80)

# Check this week data
metric_sources_this = {
    'Visits (This)': ('csv', csv_sources['newsletter'], 'Visits', 'date_range'),
    'New Subscribers (This)': ('csv', csv_sources['general_newsletter'], 'Subscribed', 'date_range'),
    'Active Unsubs (This)': ('excel', excel_sources['beehiiv_unsubscribes'], 'Active Unsubs', 'date_range'),
    'Auto Unsubs (This)': ('excel', excel_sources['beehiiv_unsubscribes'], 'Auto Unsubs', 'date_range'),
    'Unsubscribes (This)': ('excel', excel_sources['beehiiv_unsubscribes'], 'Total Unsubs', 'date_range'),
    'Net New Subscribers (This)': ('csv', csv_sources['general_newsletter'], 'Net', 'date_range'),
    'Total Subscribers (This)': ('csv', csv_sources['general_newsletter'], 'Active Subscribers', 'date_range'),
    'Growth Rate (This)': ('csv', csv_sources['general_newsletter'], 'Growth Rate', 'date_range'),
    'Blended Open Rate (This)': ('csv', csv_sources['newsletter_series_blended'], 'Open Rate', 'start_end_dates'),
    'Blended Click Rate (This)': ('csv', csv_sources['newsletter_series_blended'], 'Verified Click-Through Rate', 'start_end_dates'),
    'Blended Unsub Rate (This)': ('csv', csv_sources['newsletter_series_blended'], 'Unsubscribe Rate', 'start_end_dates'),
}

for metric_name, source_info in metric_sources_this.items():
    source_type = source_info[0]
    source_path = source_info[1]
    column_name = source_info[2]
    date_type = source_info[3]
    
    if source_type == 'csv' and date_type == 'date_range':
        diagnosis = diagnose_weekly_blank_csv(source_path, this_week_range, column_name, metric_name)
    elif source_type == 'csv' and date_type == 'start_end_dates':
        diagnosis = diagnose_weekly_blank_csv_by_dates(
            source_path, 
            week_start_date, 
            week_end_date, 
            column_name, 
            metric_name
        )
    else:  # excel
        diagnosis = diagnose_weekly_blank_excel(source_path, 'Sheet1', this_week_range, column_name, metric_name)
    
    if diagnosis['issue'] not in ['NO_ISSUE']:
        blank_metrics.append(diagnosis)
        issue_summary[diagnosis['issue']] = issue_summary.get(diagnosis['issue'], 0) + 1
        
        print(f"❌ {diagnosis['metric']}: {diagnosis['issue']}")
        print(f"   → {diagnosis['details']}\n")

print("\n" + "=" * 80)
print("DIAGNOSIS SUMMARY")
print("=" * 80)
print(f"Found {len(blank_metrics)} blank value(s)\n")

if issue_summary:
    print("Issue breakdown:")
    for issue_type, count in issue_summary.items():
        print(f"  - {issue_type}: {count} occurrence(s)")
else:
    print("✓ All values found successfully!")

print("\n" + "=" * 80)


DIAGNOSING BLANK VALUES FOR WEEKLY DATA

Checking PREVIOUS WEEK data sources:
--------------------------------------------------------------------------------
❌ Blended Open Rate (Prev): DATE_RANGE_NOT_FOUND
   → Date range '11/23/2025 - 11/29/2025' not found in file

❌ Blended Click Rate (Prev): DATE_RANGE_NOT_FOUND
   → Date range '11/23/2025 - 11/29/2025' not found in file

❌ Blended Unsub Rate (Prev): DATE_RANGE_NOT_FOUND
   → Date range '11/23/2025 - 11/29/2025' not found in file


Checking THIS WEEK data sources:
--------------------------------------------------------------------------------
❌ Blended Open Rate (This): DATE_RANGE_NOT_FOUND
   → Date range '11/30/2025 - 12/06/2025' not found in file

❌ Blended Click Rate (This): DATE_RANGE_NOT_FOUND
   → Date range '11/30/2025 - 12/06/2025' not found in file

❌ Blended Unsub Rate (This): DATE_RANGE_NOT_FOUND
   → Date range '11/30/2025 - 12/06/2025' not found in file


DIAGNOSIS SUMMARY
Found 6 blank value(s)

Issue breakdown:
 

### Calculate WoW Changes

In [6]:
# Step 5: Calculate WoW (Week-over-Week) changes for all metrics

print("\n" + "=" * 70)
print("CALCULATING WOW CHANGES")
print("=" * 70)

# Calculate change for each metric
for metric_name, values in weekly_data.items():
    wow_change = calculate_wow_change(values['this_week'], values['prev_week'])
    weekly_data[metric_name]['wow_change'] = wow_change
    print(f"{metric_name}: {wow_change}")

print("\n" + "=" * 70)
print("CALCULATIONS COMPLETE")
print("=" * 70)


CALCULATING WOW CHANGES
Visits (Newsletter Landing Page Visits): +12.35%
New Subscribers: +6.10%
Join LP CVR %: -5.59%
Active Unsubs: +559.83%
Auto Unsubs: -99.97%
Unsubscribes: -31.98%
Leak Rate: -35.89%
Net New Subscribers: -3.39%
Total Subscribers: +1.46%
List Unsub Rate: -32.86%
Growth Rate (%): -5.81%
Blended Open Rate: -1.03%
Blended Click Rate: +10.68%
Blended Unsub Rate: +13.33%

CALCULATIONS COMPLETE


### Create Weekly Comparison Table

In [7]:
# Step 6: Create the weekly comparison table as a DataFrame

print("\n" + "=" * 70)
print("CREATING WEEKLY COMPARISON TABLE")
print("=" * 70)

# Create list of dictionaries for DataFrame
table_data = []

# Define metrics in order with corrected names
metrics_order = [
    'Visits',  # Changed from 'Visits (Newsletter Landing Page Visits)'
    'CVR %',   # Changed from 'Join LP CVR %'
    'New Subscribers',
    'Active Unsubs',
    'Auto Unsubs',
    'Unsubscribes',
    'Leak Rate',
    'Net New Subscribers',
    'List Unsub Rate',
    'Total Subscribers',
    'Growth Rate (%)',
    'Blended Open Rate',
    'Blended Click Rate',
    'Blended Unsub Rate'
]

# Map to the actual keys in weekly_data dictionary
metric_key_mapping = {
    'Visits': 'Visits (Newsletter Landing Page Visits)',
    'CVR %': 'Join LP CVR %',
    'New Subscribers': 'New Subscribers',
    'Active Unsubs': 'Active Unsubs',
    'Auto Unsubs': 'Auto Unsubs',
    'Unsubscribes': 'Unsubscribes',
    'Leak Rate': 'Leak Rate',
    'Net New Subscribers': 'Net New Subscribers',
    'List Unsub Rate': 'List Unsub Rate',
    'Total Subscribers': 'Total Subscribers',
    'Growth Rate (%)': 'Growth Rate (%)',
    'Blended Open Rate': 'Blended Open Rate',
    'Blended Click Rate': 'Blended Click Rate',
    'Blended Unsub Rate': 'Blended Unsub Rate'
}

for metric_display_name in metrics_order:
    metric_key = metric_key_mapping[metric_display_name]
    table_data.append({
        'Metric': metric_display_name,
        prev_week_range: weekly_data[metric_key]['prev_week'],
        this_week_range: weekly_data[metric_key]['this_week'],
        'WoW': weekly_data[metric_key]['wow_change'],
        f'6 Wk Average ({six_week_range})': weekly_data[metric_key]['six_week_avg']
    })

# Create DataFrame
weekly_table = pd.DataFrame(table_data)

# Display the table
print("\nWEEKLY COMPARISON TABLE:")
print("=" * 70)
display(weekly_table)


CREATING WEEKLY COMPARISON TABLE

WEEKLY COMPARISON TABLE:


,Metric,11/23/2025 - 11/29/2025,11/30/2025 - 12/06/2025,WoW,6 Wk Average (10/26/2025 - 12/06/2025)
0,Visits,33821,37997,+12.35%,29825.0
1,CVR %,13.24%,12.50%,-5.59%,13.72%
2,New Subscribers,4477,4750,+6.10%,4092.67
3,Active Unsubs,356,2349,+559.83%,303.83
4,Auto Unsubs,3099,1,-99.97%,29.5
5,Unsubscribes,3455,2350,-31.98%,333.33
6,Leak Rate,77.17%,49.47%,-35.89%,8.14%
7,Net New Subscribers,2540,2454,-3.39%,1683.0
8,List Unsub Rate,2.10%,1.41%,-32.86%,0.20%
9,Total Subscribers,164839,167250,+1.46%,


### Save Weekly Table

In [8]:
# Step 7: Save the weekly comparison table to CSV and Excel

# Define output directory and file paths
output_dir = '../outputs/newsletter_analysis'
os.makedirs(output_dir, exist_ok=True)

# Save to CSV
csv_output_file = os.path.join(output_dir, 'weekly_newsletter_metrics.csv')
weekly_table.to_csv(csv_output_file, index=False)
print(f"\n✓ Weekly comparison table saved to: {csv_output_file}")

# Save to Excel with color coding
excel_output_file = os.path.join(output_dir, 'weekly_newsletter_metrics.xlsx')

wb = Workbook()
ws = wb.active
ws.title = "Weekly Comparison"

# Write headers
headers = list(weekly_table.columns)
for col_idx, header in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col_idx, value=header)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal='center', vertical='center')

# Define metrics where negative is good (reverse color coding)
reverse_color_metrics = ['Active Unsubs', 'Auto Unsubs', 'Unsubscribes', 'Leak Rate', 'List Unsub Rate']

# Write data rows
for row_idx, row_data in enumerate(weekly_table.values, start=2):
    metric_name = row_data[0]  # First column is metric name
    
    for col_idx, value in enumerate(row_data, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        
        # Apply color coding to WoW column (4th column)
        if col_idx == 4:  # WoW column
            if isinstance(value, str) and value != '':
                # Check if this metric uses reverse color coding
                if metric_name in reverse_color_metrics:
                    # Reverse: Green for negative, Red for positive
                    if value.startswith('-'):
                        cell.font = Font(color="00B050", bold=True)  # Green for negative
                    elif value.startswith('+'):
                        cell.font = Font(color="FF0000", bold=True)  # Red for positive
                else:
                    # Normal: Green for positive, Red for negative
                    if value.startswith('+'):
                        cell.font = Font(color="00B050", bold=True)  # Green
                    elif value.startswith('-'):
                        cell.font = Font(color="FF0000", bold=True)  # Red

# Adjust column widths
ws.column_dimensions['A'].width = 40  # Metric column
ws.column_dimensions['B'].width = 25  # Previous week column
ws.column_dimensions['C'].width = 25  # This week column
ws.column_dimensions['D'].width = 15  # WoW column
ws.column_dimensions['E'].width = 30  # 6 Wk Average column

# Save the workbook
wb.save(excel_output_file)

print(f"✓ Weekly comparison table with color coding saved to: {excel_output_file}")
print("\n" + "=" * 70)
print("WEEKLY TABLE GENERATION COMPLETE!")
print("=" * 70)


✓ Weekly comparison table saved to: ../outputs/newsletter_analysis/weekly_newsletter_metrics.csv
✓ Weekly comparison table with color coding saved to: ../outputs/newsletter_analysis/weekly_newsletter_metrics.xlsx

WEEKLY TABLE GENERATION COMPLETE!


#

## MONTHLY TABLE

### Load Monthly Reference Dates

In [9]:
# Step 1: Load the Custom Start Date and Custom End Date for Monthly analysis

custom_start_date = None
custom_end_date = None

# Read the analysis_ref_date.csv file
with open(ref_date_path, 'r') as f:
    lines = f.readlines()

for line in lines:
    if 'Custom Start Date' in line and custom_start_date is None:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            custom_start_date = parts[1].strip()
    
    if 'Custom End Date' in line and custom_end_date is None:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            custom_end_date = parts[1].strip()
    
    if custom_start_date and custom_end_date:
        break

print("Monthly Reference Dates Loaded:")
print(f"Custom Start Date: {custom_start_date}")
print(f"Custom End Date: {custom_end_date}")

# Create date range string - use exact format from CSV files
this_month_range = f"{custom_start_date} - {custom_end_date}"
print(f"\nThis Month Range: {this_month_range}")

# Calculate previous month
this_month_start = pd.to_datetime(custom_start_date, format='%m/%d/%Y')
this_month_end = pd.to_datetime(custom_end_date, format='%m/%d/%Y')

prev_month_start = this_month_start - pd.DateOffset(months=1)
prev_month_end = this_month_end - pd.DateOffset(months=1)

prev_month_range = f"{prev_month_start.strftime('%m/%d/%Y')} - {prev_month_end.strftime('%m/%d/%Y')}"
print(f"Previous Month Range: {prev_month_range}")

days_in_period = (this_month_end - this_month_start).days + 1
print(f"\nDays in analysis period: {days_in_period}")

Monthly Reference Dates Loaded:
Custom Start Date: 11/01/2025
Custom End Date: 11/28/2025

This Month Range: 11/01/2025 - 11/28/2025
Previous Month Range: 10/01/2025 - 10/28/2025

Days in analysis period: 28


### Define Monthly Helper Function

In [10]:
# Step 2: Define helper function for monthly calculations

def calculate_mom_change(current_value, previous_value):
    """
    Calculate Month-over-Month (MoM) or Month-to-Date (MTD) percentage change
    
    Formula: ((Current - Previous) / Previous) * 100
    """
    try:
        # Check if either value is blank/empty
        if current_value == '' or previous_value == '' or current_value is None or previous_value is None:
            return ''
        
        # Convert to float (handle percentage strings)
        if isinstance(current_value, str):
            current_value = current_value.replace('%', '').strip()
            if current_value == '':
                return ''
            current_value = float(current_value)
        
        if isinstance(previous_value, str):
            previous_value = previous_value.replace('%', '').strip()
            if previous_value == '':
                return ''
            previous_value = float(previous_value)
        
        # Avoid division by zero
        if previous_value == 0:
            return ''
        
        # Calculate percentage change
        mom_change = ((current_value - previous_value) / previous_value) * 100
        
        # Format with + or - sign
        if mom_change >= 0:
            return f"+{mom_change:.2f}%"
        else:
            return f"{mom_change:.2f}%"
    
    except Exception as e:
        print(f"Error calculating MoM change: {str(e)}")
        return ''

print("✓ Monthly helper function defined successfully!")

✓ Monthly helper function defined successfully!


### Extract Monthly Data

In [11]:
# Step 3: Extract data for both This Month and Previous Month

print("=" * 70)
print("EXTRACTING MONTHLY NEWSLETTER DATA")
print("=" * 70)

# Define all data source paths for monthly data
monthly_csv_sources = {
    'newsletter': '../outputs/newsletter/custom_newsletter.csv',
    'general_newsletter': '../outputs/general_newsletter/custom_table.csv',
    'newsletter_series_blended': '../outputs/newsletter_series/blended_custom.csv',
}

monthly_excel_sources = {
    'beehiiv_unsubscribes': '../data/beehiiv_unsubscribes.xlsx'
}

# Initialize dictionary to store monthly data
monthly_data = {}

print(f"\nThis Month: {this_month_range}")
print(f"Previous Month: {prev_month_range}")
print("\n" + "-" * 70)

# 1. Visits (from newsletter/custom_newsletter.csv)
print("\n1. Extracting Visits...")
this_visits_m = get_value_from_csv(monthly_csv_sources['newsletter'], this_month_range, 'Visits')
prev_visits_m = get_value_from_csv(monthly_csv_sources['newsletter'], prev_month_range, 'Visits')
monthly_data['Visits (Newsletter Landing Page Visits)'] = {
    'this_month': this_visits_m,
    'prev_month': prev_visits_m
}
print(f"   This Month: {this_visits_m}")
print(f"   Previous Month: {prev_visits_m}")

# 2. New Subscribers (from general_newsletter/custom_table.csv)
print("\n2. Extracting New Subscribers...")
this_new_subs_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], this_month_range, 'Subscribed')
prev_new_subs_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], prev_month_range, 'Subscribed')
monthly_data['New Subscribers'] = {
    'this_month': this_new_subs_m,
    'prev_month': prev_new_subs_m
}
print(f"   This Month: {this_new_subs_m}")
print(f"   Previous Month: {prev_new_subs_m}")

# 3. CVR % (Calculated: New Subscribers / Visits)
print("\n3. Calculating Join LP CVR %...")
this_cvr_m = calculate_division_percentage(this_new_subs_m, this_visits_m)
prev_cvr_m = calculate_division_percentage(prev_new_subs_m, prev_visits_m)
monthly_data['Join LP CVR %'] = {
    'this_month': this_cvr_m,
    'prev_month': prev_cvr_m
}
print(f"   This Month: {this_cvr_m}")
print(f"   Previous Month: {prev_cvr_m}")

# 4. Active Unsubs (from data/beehiiv_unsubscribes.xlsx)
print("\n4. Extracting Active Unsubs...")
this_active_unsubs_m = get_value_from_excel(monthly_excel_sources['beehiiv_unsubscribes'], 'Sheet1', this_month_range, 'Active Unsubs')
prev_active_unsubs_m = get_value_from_excel(monthly_excel_sources['beehiiv_unsubscribes'], 'Sheet1', prev_month_range, 'Active Unsubs')
monthly_data['Active Unsubs'] = {
    'this_month': this_active_unsubs_m,
    'prev_month': prev_active_unsubs_m
}
print(f"   This Month: {this_active_unsubs_m}")
print(f"   Previous Month: {prev_active_unsubs_m}")

# 5. Auto Unsubs (from data/beehiiv_unsubscribes.xlsx)
print("\n5. Extracting Auto Unsubs...")
this_auto_unsubs_m = get_value_from_excel(monthly_excel_sources['beehiiv_unsubscribes'], 'Sheet1', this_month_range, 'Auto Unsubs')
prev_auto_unsubs_m = get_value_from_excel(monthly_excel_sources['beehiiv_unsubscribes'], 'Sheet1', prev_month_range, 'Auto Unsubs')
monthly_data['Auto Unsubs'] = {
    'this_month': this_auto_unsubs_m,
    'prev_month': prev_auto_unsubs_m
}
print(f"   This Month: {this_auto_unsubs_m}")
print(f"   Previous Month: {prev_auto_unsubs_m}")

# 6. Unsubscribes (from data/beehiiv_unsubscribes.xlsx)
print("\n6. Extracting Unsubscribes...")
this_unsubs_m = get_value_from_excel(monthly_excel_sources['beehiiv_unsubscribes'], 'Sheet1', this_month_range, 'Total Unsubs')
prev_unsubs_m = get_value_from_excel(monthly_excel_sources['beehiiv_unsubscribes'], 'Sheet1', prev_month_range, 'Total Unsubs')
monthly_data['Unsubscribes'] = {
    'this_month': this_unsubs_m,
    'prev_month': prev_unsubs_m
}
print(f"   This Month: {this_unsubs_m}")
print(f"   Previous Month: {prev_unsubs_m}")

# 7. Leak Rate (Calculated: Unsubscribes / New Subscribers)
print("\n7. Calculating Leak Rate...")
this_leak_rate_m = calculate_division_percentage(this_unsubs_m, this_new_subs_m)
prev_leak_rate_m = calculate_division_percentage(prev_unsubs_m, prev_new_subs_m)
monthly_data['Leak Rate'] = {
    'this_month': this_leak_rate_m,
    'prev_month': prev_leak_rate_m
}
print(f"   This Month: {this_leak_rate_m}")
print(f"   Previous Month: {prev_leak_rate_m}")

# 8. Net New Subscribers (from general_newsletter/custom_table.csv)
print("\n8. Extracting Net New Subscribers...")
this_net_new_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], this_month_range, 'Net')
prev_net_new_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], prev_month_range, 'Net')
monthly_data['Net New Subscribers'] = {
    'this_month': this_net_new_m,
    'prev_month': prev_net_new_m
}
print(f"   This Month: {this_net_new_m}")
print(f"   Previous Month: {prev_net_new_m}")

# 9. Total Subscribers (from general_newsletter/custom_table.csv)
print("\n9. Extracting Total Subscribers...")
this_total_subs_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], this_month_range, 'Active Subscribers')
prev_total_subs_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], prev_month_range, 'Active Subscribers')
monthly_data['Total Subscribers'] = {
    'this_month': this_total_subs_m,
    'prev_month': prev_total_subs_m
}
print(f"   This Month: {this_total_subs_m}")
print(f"   Previous Month: {prev_total_subs_m}")

# 10. List Unsub Rate (Calculated: Unsubscribes / Total Subscribers)
print("\n10. Calculating List Unsub Rate...")
this_list_unsub_rate_m = calculate_division_percentage(this_unsubs_m, this_total_subs_m)
prev_list_unsub_rate_m = calculate_division_percentage(prev_unsubs_m, prev_total_subs_m)
monthly_data['List Unsub Rate'] = {
    'this_month': this_list_unsub_rate_m,
    'prev_month': prev_list_unsub_rate_m
}
print(f"   This Month: {this_list_unsub_rate_m}")
print(f"   Previous Month: {prev_list_unsub_rate_m}")

# 11. Growth Rate (from general_newsletter/custom_table.csv - convert decimal to percentage)
print("\n11. Extracting Growth Rate (%)...")
this_growth_raw_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], this_month_range, 'Growth Rate')
prev_growth_raw_m = get_value_from_csv(monthly_csv_sources['general_newsletter'], prev_month_range, 'Growth Rate')
this_growth_m = convert_decimal_to_percentage(this_growth_raw_m)
prev_growth_m = convert_decimal_to_percentage(prev_growth_raw_m)
monthly_data['Growth Rate (%)'] = {
    'this_month': this_growth_m,
    'prev_month': prev_growth_m
}
print(f"   This Month: {this_growth_m}")
print(f"   Previous Month: {prev_growth_m}")

# 12. Blended Open Rate (from newsletter_series/blended_custom.csv)
# NOTE: Need to check if this CSV uses 'Date Range' or separate date columns
print("\n12. Extracting Blended Open Rate...")
# Try with Date Range first
this_open_rate_m = get_value_from_csv(monthly_csv_sources['newsletter_series_blended'], this_month_range, 'Open Rate')
prev_open_rate_m = get_value_from_csv(monthly_csv_sources['newsletter_series_blended'], prev_month_range, 'Open Rate')

# If blank, try with Start/End dates
if this_open_rate_m == '':
    this_open_rate_m = get_value_from_csv_by_dates(
        monthly_csv_sources['newsletter_series_blended'], 
        custom_start_date, 
        custom_end_date, 
        'Open Rate'
    )
if prev_open_rate_m == '':
    prev_open_rate_m = get_value_from_csv_by_dates(
        monthly_csv_sources['newsletter_series_blended'], 
        prev_month_start.strftime('%m/%d/%Y'), 
        prev_month_end.strftime('%m/%d/%Y'), 
        'Open Rate'
    )

monthly_data['Blended Open Rate'] = {
    'this_month': this_open_rate_m,
    'prev_month': prev_open_rate_m
}
print(f"   This Month: {this_open_rate_m}")
print(f"   Previous Month: {prev_open_rate_m}")

# 13. Blended Click Rate (from newsletter_series/blended_custom.csv)
print("\n13. Extracting Blended Click Rate...")
# Try with Date Range first
this_click_rate_m = get_value_from_csv(monthly_csv_sources['newsletter_series_blended'], this_month_range, 'Verified Click-Through Rate')
prev_click_rate_m = get_value_from_csv(monthly_csv_sources['newsletter_series_blended'], prev_month_range, 'Verified Click-Through Rate')

# If blank, try with Start/End dates
if this_click_rate_m == '':
    this_click_rate_m = get_value_from_csv_by_dates(
        monthly_csv_sources['newsletter_series_blended'], 
        custom_start_date, 
        custom_end_date, 
        'Verified Click-Through Rate'
    )
if prev_click_rate_m == '':
    prev_click_rate_m = get_value_from_csv_by_dates(
        monthly_csv_sources['newsletter_series_blended'], 
        prev_month_start.strftime('%m/%d/%Y'), 
        prev_month_end.strftime('%m/%d/%Y'), 
        'Verified Click-Through Rate'
    )

monthly_data['Blended Click Rate'] = {
    'this_month': this_click_rate_m,
    'prev_month': prev_click_rate_m
}
print(f"   This Month: {this_click_rate_m}")
print(f"   Previous Month: {prev_click_rate_m}")

# 14. Blended Unsub Rate (from newsletter_series/blended_custom.csv)
print("\n14. Extracting Blended Unsub Rate...")
# Try with Date Range first
this_blended_unsub_m = get_value_from_csv(monthly_csv_sources['newsletter_series_blended'], this_month_range, 'Unsubscribe Rate')
prev_blended_unsub_m = get_value_from_csv(monthly_csv_sources['newsletter_series_blended'], prev_month_range, 'Unsubscribe Rate')

# If blank, try with Start/End dates
if this_blended_unsub_m == '':
    this_blended_unsub_m = get_value_from_csv_by_dates(
        monthly_csv_sources['newsletter_series_blended'], 
        custom_start_date, 
        custom_end_date, 
        'Unsubscribe Rate'
    )
if prev_blended_unsub_m == '':
    prev_blended_unsub_m = get_value_from_csv_by_dates(
        monthly_csv_sources['newsletter_series_blended'], 
        prev_month_start.strftime('%m/%d/%Y'), 
        prev_month_end.strftime('%m/%d/%Y'), 
        'Unsubscribe Rate'
    )

monthly_data['Blended Unsub Rate'] = {
    'this_month': this_blended_unsub_m,
    'prev_month': prev_blended_unsub_m
}
print(f"   This Month: {this_blended_unsub_m}")
print(f"   Previous Month: {prev_blended_unsub_m}")

EXTRACTING MONTHLY NEWSLETTER DATA

This Month: 11/01/2025 - 11/28/2025
Previous Month: 10/01/2025 - 10/28/2025

----------------------------------------------------------------------

1. Extracting Visits...
   This Month: 120443
   Previous Month: 60443

2. Extracting New Subscribers...
   This Month: 16289
   Previous Month: 11488

3. Calculating Join LP CVR %...
   This Month: 13.52%
   Previous Month: 19.01%

4. Extracting Active Unsubs...
   This Month: 5467
   Previous Month: 4321

5. Extracting Auto Unsubs...
   This Month: 4478
   Previous Month: 1357

6. Extracting Unsubscribes...
   This Month: 9945
   Previous Month: 5678

7. Calculating Leak Rate...
   This Month: 61.05%
   Previous Month: 49.43%

8. Extracting Net New Subscribers...
   This Month: 6010
   Previous Month: 2848

9. Extracting Total Subscribers...
   This Month: 164354
   Previous Month: 158149

10. Calculating List Unsub Rate...
   This Month: 6.05%
   Previous Month: 3.59%

11. Extracting Growth Rate (%)..

### Diagnose Blank Values

In [12]:
# Step 3.5: Diagnose why monthly values might be blank

print(f"\n{'='*80}")
print("DIAGNOSING BLANK VALUES FOR MONTHLY DATA")
print(f"{'='*80}\n")

# Check metrics with blank values
blank_metrics_monthly = []
issue_summary_monthly = {}

print("Checking PREVIOUS MONTH data sources:")
print("-" * 80)

# Map metrics to their sources
# NOTE: Monthly blended files use 'Date Range' column, NOT 'Start Date'/'End Date'
monthly_metric_sources = {
    'Visits (Prev)': ('csv', monthly_csv_sources['newsletter'], 'Visits', 'date_range'),
    'New Subscribers (Prev)': ('csv', monthly_csv_sources['general_newsletter'], 'Subscribed', 'date_range'),
    'Active Unsubs (Prev)': ('excel', monthly_excel_sources['beehiiv_unsubscribes'], 'Active Unsubs', 'date_range'),
    'Auto Unsubs (Prev)': ('excel', monthly_excel_sources['beehiiv_unsubscribes'], 'Auto Unsubs', 'date_range'),
    'Unsubscribes (Prev)': ('excel', monthly_excel_sources['beehiiv_unsubscribes'], 'Total Unsubs', 'date_range'),
    'Net New Subscribers (Prev)': ('csv', monthly_csv_sources['general_newsletter'], 'Net', 'date_range'),
    'Total Subscribers (Prev)': ('csv', monthly_csv_sources['general_newsletter'], 'Active Subscribers', 'date_range'),
    'Growth Rate (Prev)': ('csv', monthly_csv_sources['general_newsletter'], 'Growth Rate', 'date_range'),
    'Blended Open Rate (Prev)': ('csv', monthly_csv_sources['newsletter_series_blended'], 'Open Rate', 'date_range'),  # Changed to 'date_range'
    'Blended Click Rate (Prev)': ('csv', monthly_csv_sources['newsletter_series_blended'], 'Verified Click-Through Rate', 'date_range'),  # Changed to 'date_range'
    'Blended Unsub Rate (Prev)': ('csv', monthly_csv_sources['newsletter_series_blended'], 'Unsubscribe Rate', 'date_range'),  # Changed to 'date_range'
}

for metric_name, source_info in monthly_metric_sources.items():
    source_type = source_info[0]
    source_path = source_info[1]
    column_name = source_info[2]
    date_type = source_info[3]
    
    if source_type == 'csv' and date_type == 'date_range':
        diagnosis = diagnose_weekly_blank_csv(source_path, prev_month_range, column_name, metric_name)
    elif source_type == 'csv' and date_type == 'start_end_dates':
        diagnosis = diagnose_weekly_blank_csv_by_dates(
            source_path, 
            prev_month_start.strftime('%m/%d/%Y'), 
            prev_month_end.strftime('%m/%d/%Y'), 
            column_name, 
            metric_name
        )
    else:  # excel
        diagnosis = diagnose_weekly_blank_excel(source_path, 'Sheet1', prev_month_range, column_name, metric_name)
    
    if diagnosis['issue'] not in ['NO_ISSUE']:
        blank_metrics_monthly.append(diagnosis)
        issue_summary_monthly[diagnosis['issue']] = issue_summary_monthly.get(diagnosis['issue'], 0) + 1
        
        print(f"❌ {diagnosis['metric']}: {diagnosis['issue']}")
        print(f"   → {diagnosis['details']}\n")

print("\n" + "=" * 80)
print("Checking THIS MONTH data sources:")
print("-" * 80)

# Check this month data
monthly_metric_sources_this = {
    'Visits (This)': ('csv', monthly_csv_sources['newsletter'], 'Visits', 'date_range'),
    'New Subscribers (This)': ('csv', monthly_csv_sources['general_newsletter'], 'Subscribed', 'date_range'),
    'Active Unsubs (This)': ('excel', monthly_excel_sources['beehiiv_unsubscribes'], 'Active Unsubs', 'date_range'),
    'Auto Unsubs (This)': ('excel', monthly_excel_sources['beehiiv_unsubscribes'], 'Auto Unsubs', 'date_range'),
    'Unsubscribes (This)': ('excel', monthly_excel_sources['beehiiv_unsubscribes'], 'Total Unsubs', 'date_range'),
    'Net New Subscribers (This)': ('csv', monthly_csv_sources['general_newsletter'], 'Net', 'date_range'),
    'Total Subscribers (This)': ('csv', monthly_csv_sources['general_newsletter'], 'Active Subscribers', 'date_range'),
    'Growth Rate (This)': ('csv', monthly_csv_sources['general_newsletter'], 'Growth Rate', 'date_range'),
    'Blended Open Rate (This)': ('csv', monthly_csv_sources['newsletter_series_blended'], 'Open Rate', 'date_range'),  # Changed to 'date_range'
    'Blended Click Rate (This)': ('csv', monthly_csv_sources['newsletter_series_blended'], 'Verified Click-Through Rate', 'date_range'),  # Changed to 'date_range'
    'Blended Unsub Rate (This)': ('csv', monthly_csv_sources['newsletter_series_blended'], 'Unsubscribe Rate', 'date_range'),  # Changed to 'date_range'
}

for metric_name, source_info in monthly_metric_sources_this.items():
    source_type = source_info[0]
    source_path = source_info[1]
    column_name = source_info[2]
    date_type = source_info[3]
    
    if source_type == 'csv' and date_type == 'date_range':
        diagnosis = diagnose_weekly_blank_csv(source_path, this_month_range, column_name, metric_name)
    elif source_type == 'csv' and date_type == 'start_end_dates':
        diagnosis = diagnose_weekly_blank_csv_by_dates(
            source_path, 
            custom_start_date, 
            custom_end_date, 
            column_name, 
            metric_name
        )
    else:  # excel
        diagnosis = diagnose_weekly_blank_excel(source_path, 'Sheet1', this_month_range, column_name, metric_name)
    
    if diagnosis['issue'] not in ['NO_ISSUE']:
        blank_metrics_monthly.append(diagnosis)
        issue_summary_monthly[diagnosis['issue']] = issue_summary_monthly.get(diagnosis['issue'], 0) + 1
        
        print(f"❌ {diagnosis['metric']}: {diagnosis['issue']}")
        print(f"   → {diagnosis['details']}\n")

print("\n" + "=" * 80)
print("DIAGNOSIS SUMMARY")
print("=" * 80)
print(f"Found {len(blank_metrics_monthly)} blank value(s)\n")

if issue_summary_monthly:
    print("Issue breakdown:")
    for issue_type, count in issue_summary_monthly.items():
        print(f"  - {issue_type}: {count} occurrence(s)")
else:
    print("✓ All values found successfully!")

print("\n" + "=" * 80)


DIAGNOSING BLANK VALUES FOR MONTHLY DATA

Checking PREVIOUS MONTH data sources:
--------------------------------------------------------------------------------

Checking THIS MONTH data sources:
--------------------------------------------------------------------------------

DIAGNOSIS SUMMARY
Found 0 blank value(s)

✓ All values found successfully!



### Calculate MoM Changes

In [13]:
# Step 4: Calculate MoM/MTD Change for all metrics

print("\n" + "=" * 70)
print("CALCULATING MOM/MTD CHANGES")
print("=" * 70)

# Calculate change for each metric
for metric_name, values in monthly_data.items():
    mom_change = calculate_mom_change(values['this_month'], values['prev_month'])
    monthly_data[metric_name]['mom_change'] = mom_change
    print(f"{metric_name}: {mom_change}")

print("\n" + "=" * 70)
print("CALCULATIONS COMPLETE")
print("=" * 70)


CALCULATING MOM/MTD CHANGES
Visits (Newsletter Landing Page Visits): +99.27%
New Subscribers: +41.79%
Join LP CVR %: -28.88%
Active Unsubs: +26.52%
Auto Unsubs: +229.99%
Unsubscribes: +75.15%
Leak Rate: +23.51%
Net New Subscribers: +111.03%
Total Subscribers: +3.92%
List Unsub Rate: +68.52%
Growth Rate (%): +108.05%
Blended Open Rate: -1.27%
Blended Click Rate: -11.43%
Blended Unsub Rate: -11.76%

CALCULATIONS COMPLETE


### Create Monthly Comparison Table

In [14]:
# Step 5: Create the monthly comparison table

print("\n" + "=" * 70)
print("CREATING MONTHLY COMPARISON TABLE")
print("=" * 70)

table_data_monthly = []

# Define metrics in order with corrected names
monthly_metrics_order = [
    'Visits',  # Changed from 'Visits (Newsletter Landing Page Visits)'
    'CVR %',   # Changed from 'Join LP CVR %'
    'New Subscribers',
    'Active Unsubs',
    'Auto Unsubs',
    'Unsubscribes',
    'Leak Rate',
    'Net New Subscribers',
    'List Unsub Rate',
    'Total Subscribers',
    'Growth Rate (%)',
    'Blended Open Rate',
    'Blended Click Rate',
    'Blended Unsub Rate'
]

# Map to the actual keys in monthly_data dictionary
monthly_metric_key_mapping = {
    'Visits': 'Visits (Newsletter Landing Page Visits)',
    'CVR %': 'Join LP CVR %',
    'New Subscribers': 'New Subscribers',
    'Active Unsubs': 'Active Unsubs',
    'Auto Unsubs': 'Auto Unsubs',
    'Unsubscribes': 'Unsubscribes',
    'Leak Rate': 'Leak Rate',
    'Net New Subscribers': 'Net New Subscribers',
    'List Unsub Rate': 'List Unsub Rate',
    'Total Subscribers': 'Total Subscribers',
    'Growth Rate (%)': 'Growth Rate (%)',
    'Blended Open Rate': 'Blended Open Rate',
    'Blended Click Rate': 'Blended Click Rate',
    'Blended Unsub Rate': 'Blended Unsub Rate'
}

for metric_display_name in monthly_metrics_order:
    metric_key = monthly_metric_key_mapping[metric_display_name]
    table_data_monthly.append({
        'Metric': metric_display_name,
        prev_month_range: monthly_data[metric_key]['prev_month'],
        this_month_range: monthly_data[metric_key]['this_month'],
        'MoM': monthly_data[metric_key]['mom_change']
    })

# Create DataFrame
monthly_table = pd.DataFrame(table_data_monthly)

# Display the table
print("\nMONTHLY COMPARISON TABLE:")
print("=" * 70)
display(monthly_table)


CREATING MONTHLY COMPARISON TABLE

MONTHLY COMPARISON TABLE:


,Metric,10/01/2025 - 10/28/2025,11/01/2025 - 11/28/2025,MoM
0,Visits,60443,120443,+99.27%
1,CVR %,19.01%,13.52%,-28.88%
2,New Subscribers,11488,16289,+41.79%
3,Active Unsubs,4321,5467,+26.52%
4,Auto Unsubs,1357,4478,+229.99%
5,Unsubscribes,5678,9945,+75.15%
6,Leak Rate,49.43%,61.05%,+23.51%
7,Net New Subscribers,2848,6010,+111.03%
8,List Unsub Rate,3.59%,6.05%,+68.52%
9,Total Subscribers,158149,164354,+3.92%


### Save Monthly Table

In [15]:
# Step 6: Save the monthly comparison table to CSV and Excel

# Save to CSV
csv_output_file_monthly = os.path.join(output_dir, 'monthly_newsletter_metrics.csv')
monthly_table.to_csv(csv_output_file_monthly, index=False)
print(f"\n✓ Monthly comparison table saved to: {csv_output_file_monthly}")

# Save to Excel with color coding
excel_output_file_monthly = os.path.join(output_dir, 'monthly_newsletter_metrics.xlsx')

wb_monthly = Workbook()
ws_monthly = wb_monthly.active
ws_monthly.title = "Monthly Comparison"

# Write headers
headers_monthly = list(monthly_table.columns)
for col_idx, header in enumerate(headers_monthly, start=1):
    cell = ws_monthly.cell(row=1, column=col_idx, value=header)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal='center', vertical='center')

# Define metrics where negative is good (reverse color coding)
reverse_color_metrics = ['Active Unsubs', 'Auto Unsubs', 'Unsubscribes', 'Leak Rate', 'List Unsub Rate']

# Write data rows
for row_idx, row_data in enumerate(monthly_table.values, start=2):
    metric_name = row_data[0]  # First column is metric name
    
    for col_idx, value in enumerate(row_data, start=1):
        cell = ws_monthly.cell(row=row_idx, column=col_idx, value=value)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        
        # Apply color coding to MoM column (last column)
        if col_idx == len(headers_monthly):
            if isinstance(value, str) and value != '':
                # Check if this metric uses reverse color coding
                if metric_name in reverse_color_metrics:
                    # Reverse: Green for negative, Red for positive
                    if value.startswith('-'):
                        cell.font = Font(color="00B050", bold=True)  # Green for negative
                    elif value.startswith('+'):
                        cell.font = Font(color="FF0000", bold=True)  # Red for positive
                else:
                    # Normal: Green for positive, Red for negative
                    if value.startswith('+'):
                        cell.font = Font(color="00B050", bold=True)  # Green
                    elif value.startswith('-'):
                        cell.font = Font(color="FF0000", bold=True)  # Red

# Adjust column widths
ws_monthly.column_dimensions['A'].width = 40
ws_monthly.column_dimensions['B'].width = 25
ws_monthly.column_dimensions['C'].width = 25
ws_monthly.column_dimensions['D'].width = 15

wb_monthly.save(excel_output_file_monthly)

print(f"✓ Monthly comparison table with color coding saved to: {excel_output_file_monthly}")
print("\n" + "=" * 70)
print("MONTHLY TABLE GENERATION COMPLETE!")
print("=" * 70)


✓ Monthly comparison table saved to: ../outputs/newsletter_analysis/monthly_newsletter_metrics.csv
✓ Monthly comparison table with color coding saved to: ../outputs/newsletter_analysis/monthly_newsletter_metrics.xlsx

MONTHLY TABLE GENERATION COMPLETE!
